# PDF Combiner — Iterations 4–6 (LaTeX-style sub-figure layout)

For each `(language, embedding model, ML model)` group in
`Results/_iteration_4`, `_iteration_5`, `_iteration_6`, this notebook produces a
**single A4 page** that mimics a LaTeX `subfigure` block:

- **2×2 grid** of the four plots — Confusion Matrix, Per-Class Metrics,
  Precision-Recall Curve, ROC Curve — each with a `(a)`–`(d)` sub-caption.
- **Figure title** on top (language · embedding · classifier · iteration).
- **Figure caption** at the bottom (`Figure N: …`).
- All plots are embedded as **vector** content via `Page.show_pdf_page`, so the
  output stays crisp at any zoom and is ideal for `\includegraphics` in your
  Overleaf Appendix.

Output: `Results/_iteration_<N>/Combined/<language>/<embedding>__<ml_model>.pdf`

Requires **PyMuPDF** (`pip install pymupdf`).


In [11]:
# Install PyMuPDF if missing (uncomment if needed)
# %pip install --quiet pymupdf

import fitz  # PyMuPDF
import re
import shutil
from pathlib import Path
from collections import defaultdict

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
REPO_ROOT = Path("/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026")
RESULTS_DIR = REPO_ROOT / "Results"
ITERATIONS = ["_iteration_4", "_iteration_5", "_iteration_6"]
COMBINED_SUBDIR = "Combined"

# Page geometry. Width = A4 width (so output is "A4-friendly" for LaTeX),
# but the page height is computed so the page is trimmed to its content
# (no whitespace below the plots). All plots stay vector -> max quality.
PAGE_W = 595.276        # A4 width in PDF points
PAGE_H_MAX = 841.890    # never exceed A4 height
MARGIN = 36             # uniform page margin
GUTTER_X = 14           # horizontal gap between columns
ROW_GAP  = 18           # vertical gap between rows
CAP_GAP  = 6            # gap between plot bottom and its caption
CAP_SIZE = 9            # caption font size (pt)

BODY_FONT = "helvetica"

PLOT_PREFIXES = [
    ("confusion_matrix_", "Confusion Matrix"),
    ("metrics_bar_",      "Per-Class Metrics"),
    ("precision_recall_", "Precision-Recall Curve"),
    ("pr_curve_",         "Precision-Recall Curve"),
    ("roc_curve_",        "ROC Curve"),
]

ML_SUFFIXES = [
    "_Random_Forest_Optimized", "_LightGBM_Optimized", "_XGBoost_Optimized",
    "_Voting_Ensemble", "_Random_Forest", "_LightGBM", "_XGBoost", "_SVM",
    "_random_forest", "_svm_linear", "_svm_rbf", "_lightgbm", "_xgboost",
]

SKIP_PATTERNS = ("comparison_", "best_macro_f1", "precision_recall_scatter")

# Normalise inconsistent language tokens (esp. iteration_4 used country names).
LANGUAGE_ALIASES = {
    "netherlands": "dutch",
    "germany":     "german",
    "sweden":      "swedish",
}

# Required display order: CM -> Metrics Bar -> Precision-Recall -> ROC.
PLOT_ORDER = [
    "Confusion Matrix",
    "Per-Class Metrics",
    "Precision-Recall Curve",
    "ROC Curve",
]


# ---------------------------------------------------------------------------
# Filename parsing & grouping
# ---------------------------------------------------------------------------
def parse_filename(stem: str):
    plot_label = None
    rest = None
    for prefix, label in PLOT_PREFIXES:
        if stem.startswith(prefix):
            plot_label = label
            rest = stem[len(prefix):]
            break
    if rest is None:
        return None

    ml_model = None
    for suf in ML_SUFFIXES:
        if rest.endswith(suf):
            ml_model = suf.lstrip("_")
            rest = rest[: -len(suf)]
            break
    if ml_model is None:
        return None

    if "_manual_" not in rest:
        return None
    language, embedding = rest.split("_manual_", 1)
    language = LANGUAGE_ALIASES.get(language.lower(), language)
    return plot_label, language, embedding, ml_model


def safe_name(s: str) -> str:
    return re.sub(r"[^\w\-.]+", "_", s).strip("_")


def collect_groups(iter_dir: Path):
    groups: dict = defaultdict(dict)
    skipped = 0
    for pdf in sorted(iter_dir.glob("*.pdf")):
        if any(pdf.name.startswith(p) for p in SKIP_PATTERNS):
            skipped += 1
            continue
        parsed = parse_filename(pdf.stem)
        if parsed is None:
            skipped += 1
            continue
        plot_label, lang, embed, ml = parsed
        groups[(lang, embed, ml)].setdefault(plot_label, pdf)
    return groups, skipped


# ---------------------------------------------------------------------------
# Layout & rendering
# ---------------------------------------------------------------------------
def _grid_cols(n: int) -> int:
    return 1 if n <= 1 else 2  # 1 plot -> 1 col, otherwise 2 cols


def _natural_size(src_path: Path):
    """Return (width, height) of the first page of a PDF, in points."""
    with fitz.open(src_path) as src:
        r = src[0].rect
        return r.width, r.height


def render_group_pdf(out_path: Path, plots: dict) -> bool:
    """Build a single, height-trimmed page containing only the available plots.
    Returns True on success."""
    available = [(label, plots[label]) for label in PLOT_ORDER if label in plots]
    if not available:
        return False

    cols = _grid_cols(len(available))
    rows = (len(available) + cols - 1) // cols

    cell_w = (PAGE_W - 2 * MARGIN - GUTTER_X * (cols - 1)) / cols

    # Pre-compute the rendered (scaled) height of each plot at width = cell_w.
    # We never upscale (scale capped at 1.0) so vector quality is preserved.
    rendered = []
    for label, src_path in available:
        sw, sh = _natural_size(src_path)
        scale = min(cell_w / sw, 1.0)
        rendered.append({
            "label": label,
            "src": src_path,
            "w": sw * scale,
            "h": sh * scale,
        })

    # Row height = tallest plot in row + caption + gap.
    cap_total = CAP_GAP + CAP_SIZE  # space reserved for caption beneath plot
    row_heights = []
    for r in range(rows):
        row_items = rendered[r * cols:(r + 1) * cols]
        row_heights.append(max(it["h"] for it in row_items) + cap_total)

    # Total page height: margin + rows + inter-row gaps + bottom margin.
    content_h = sum(row_heights) + ROW_GAP * (rows - 1)
    page_h = min(content_h + 2 * MARGIN, PAGE_H_MAX)

    # If we hit the cap, scale every plot uniformly so it still fits.
    if content_h + 2 * MARGIN > PAGE_H_MAX:
        avail = PAGE_H_MAX - 2 * MARGIN - ROW_GAP * (rows - 1) - cap_total * rows
        natural_row_max = sum(max(it["h"] for it in rendered[r*cols:(r+1)*cols])
                              for r in range(rows))
        shrink = avail / natural_row_max
        for it in rendered:
            it["w"] *= shrink
            it["h"] *= shrink
        row_heights = []
        for r in range(rows):
            row_items = rendered[r * cols:(r + 1) * cols]
            row_heights.append(max(it["h"] for it in row_items) + cap_total)
        page_h = PAGE_H_MAX

    # Build the document.
    doc = fitz.open()
    try:
        page = doc.new_page(width=PAGE_W, height=page_h)
        sub_letters = ["(a)", "(b)", "(c)", "(d)"]

        y_cursor = MARGIN
        for r in range(rows):
            row_items = rendered[r * cols:(r + 1) * cols]
            row_h = row_heights[r]
            for c, it in enumerate(row_items):
                idx = r * cols + c
                # Column slot is `cell_w` wide; center plot horizontally inside it.
                col_x0 = MARGIN + c * (cell_w + GUTTER_X)
                px = col_x0 + (cell_w - it["w"]) / 2
                # Anchor plot to the *top* of the row so caption is just below it.
                py = y_cursor
                rect = fitz.Rect(px, py, px + it["w"], py + it["h"])
                with fitz.open(it["src"]) as src:
                    page.show_pdf_page(rect, src, 0)

                # Caption directly under the plot.
                sub = f"{sub_letters[idx]} {it['label']}."
                tw = fitz.get_text_length(sub, fontname=BODY_FONT, fontsize=CAP_SIZE)
                cap_x = col_x0 + (cell_w - tw) / 2
                cap_y = py + it["h"] + CAP_GAP + CAP_SIZE - 2
                page.insert_text((cap_x, cap_y), sub,
                                 fontname=BODY_FONT, fontsize=CAP_SIZE)

            y_cursor += row_h + ROW_GAP

        doc.save(out_path, deflate=True, garbage=4)
        return True
    finally:
        doc.close()


# ---------------------------------------------------------------------------
# Driver
# ---------------------------------------------------------------------------
def combine_iteration(iteration: str) -> None:
    iter_dir = RESULTS_DIR / iteration
    if not iter_dir.is_dir():
        print(f"[skip] {iter_dir} does not exist")
        return

    # Wipe any previously combined PDFs so each run starts fresh.
    out_dir = iter_dir / COMBINED_SUBDIR
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    groups, skipped = collect_groups(iter_dir)
    print(f"\n=== {iteration} ===")
    print(f"  groups : {len(groups)}")
    print(f"  skipped: {skipped} (aggregate / unparseable / non-pdf)")

    written = 0
    for (lang, embed, ml), plots in sorted(groups.items()):
        if not plots:
            continue
        out_path = out_dir / f"{safe_name(lang)}__{safe_name(embed)}__{safe_name(ml)}.pdf"
        if render_group_pdf(out_path, plots):
            written += 1

    print(f"  wrote  : {written} combined PDFs -> {out_dir.relative_to(REPO_ROOT)}")


def main():
    for it in ITERATIONS:
        combine_iteration(it)
    print("\nDone.")


main()



=== _iteration_4 ===
  groups : 160
  skipped: 4 (aggregate / unparseable / non-pdf)
  wrote  : 160 combined PDFs -> Results/_iteration_4/Combined

=== _iteration_5 ===
  groups : 256
  skipped: 0 (aggregate / unparseable / non-pdf)
  wrote  : 256 combined PDFs -> Results/_iteration_5/Combined

=== _iteration_6 ===
  groups : 120
  skipped: 2 (aggregate / unparseable / non-pdf)
  wrote  : 120 combined PDFs -> Results/_iteration_6/Combined

Done.
